# **Prediksi Praktikum Klasifikasi - IF5100 Pemrograman untuk Data Analitik**
## Random Forest, KNN, Regresi Logistik, ANN, dan Ensemble

Nama : ______________________

NIM  : ______________________

### Aturan / Rules

- Kerjakan di **Google Colab** (runtime CPU). Jalankan sel **berurutan** dari atas ke bawah.
- Library: `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`. Dataset dibuat atau di-*load* langsung di notebook, tanpa file CSV.
- Gunakan `random_state=42` di **setiap** fungsi yang punya parameter tersebut (`train_test_split`, model, `StratifiedKFold`).
- **Jangan ubah** sel *setup dataset* dan nama variabel yang diminta soal.
- Setiap grafik wajib punya **judul, label sumbu, dan legend** (jika ada lebih dari satu seri).
- Pertanyaan bertanda ✍️ dijawab di sel teks (Markdown) di bawahnya.

*Work in Google Colab, run cells in order, use `random_state=42` everywhere, do not change the setup cells,
and answer the ✍️ questions in the Markdown cell below them.*

| Soal | Topik | Poin |
|---|---|---|
| 1 | Churn pelanggan internet: cleaning, `Pipeline` + `ColumnTransformer`, Regresi Logistik vs KNN vs Random Forest, threshold, interpretasi | 40 |
| 2 | Klasifikasi wine: KNN dengan/tanpa scaling, memilih *k* dengan CV, ANN (`MLPClassifier`), `GridSearchCV` | 25 |
| 3 | Diagnosis kanker payudara: ensemble (Voting, Bagging, Random Forest, Boosting, Stacking) | 25 |
| 4 | Soal teori | 10 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (RandomForestClassifier, VotingClassifier, StackingClassifier, BaggingClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.inspection import permutation_importance, DecisionBoundaryDisplay
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

---
## SOAL 1 — Churn Pelanggan Internet Rumah (40 poin)

Penyedia internet rumah **NusaNet** kehilangan banyak pelanggan setiap bulan. Tim retensi ingin model yang dapat
menandai pelanggan yang berisiko **berhenti berlangganan (churn)** agar bisa dihubungi lebih dulu. Data 3.000-an
pelanggan diambil dari dua sistem yang berbeda sehingga masih **kotor**: penulisan kota tidak seragam, ada umur yang
mustahil, ada nilai kosong, dan ada baris ganda.

*NusaNet, a home internet provider, wants a model that flags customers who are likely to churn. The data comes from
two systems and is dirty: inconsistent city names, impossible ages, missing values, and duplicate rows.*

| Kolom | Keterangan |
|---|---|
| `customer_id` | ID pelanggan |
| `gender`, `umur`, `kota` | data demografis (`kota` belum seragam, `umur` ada yang mustahil) |
| `paket` | Basic / Standard / Premium |
| `kontrak` | Bulanan / Tahunan / 2 Tahun |
| `lama_berlangganan` | lama berlangganan (bulan) |
| `biaya_bulanan` | tagihan per bulan (ribu rupiah) |
| `kecepatan_mbps` | kecepatan paket |
| `gangguan_per_bulan` | rata-rata gangguan jaringan per bulan |
| `tiket_komplain` | jumlah tiket komplain 3 bulan terakhir |
| `metode_bayar` | Auto-debit / Transfer / E-wallet |
| `churn` | **target**: Ya / Tidak |

### Setup dataset (jangan diubah)

In [ ]:
def buat_data_churn(n=3000, seed=42):
    rng = np.random.default_rng(seed)
    kontrak = rng.choice(["Bulanan", "Tahunan", "2 Tahun"], n, p=[0.5, 0.3, 0.2])
    paket = rng.choice(["Basic", "Standard", "Premium"], n, p=[0.4, 0.4, 0.2])
    lama = np.where(kontrak == "Bulanan", rng.integers(1, 36, n), rng.integers(6, 73, n))
    biaya = np.select([paket == "Basic", paket == "Standard"], [250, 400], 650) + rng.normal(0, 40, n)
    kecepatan = np.select([paket == "Basic", paket == "Standard"], [20, 50], 100)
    gangguan = rng.poisson(1.2, n)
    komplain = rng.poisson(0.3 + 0.4 * gangguan)
    metode = rng.choice(["Auto-debit", "Transfer", "E-wallet"], n, p=[0.35, 0.35, 0.30])
    z = (-2.0 + 1.6 * (kontrak == "Bulanan") + 0.3 * (kontrak == "Tahunan") - 0.035 * lama
         + 0.35 * gangguan + 0.45 * komplain + 0.002 * (biaya - 400) - 0.6 * (metode == "Auto-debit")
         + rng.normal(0, 0.5, n))
    churn = np.where(rng.random(n) < 1 / (1 + np.exp(-z)), "Ya", "Tidak")
    df = pd.DataFrame({
        "customer_id": [f"C{i:04d}" for i in range(1, n + 1)],
        "gender": rng.choice(["Laki-laki", "Perempuan"], n),
        "umur": rng.integers(18, 70, n).astype(float),
        "kota": rng.choice(["Jakarta", "Bandung", "Surabaya", "Medan"], n),
        "paket": paket, "kontrak": kontrak, "lama_berlangganan": lama,
        "biaya_bulanan": biaya.round(1), "kecepatan_mbps": kecepatan,
        "gangguan_per_bulan": gangguan, "tiket_komplain": komplain,
        "metode_bayar": metode, "churn": churn})
    # --- data kotor ---
    i = rng.choice(n, 450, replace=False)
    df.loc[i[:150], "kota"] = df.loc[i[:150], "kota"].str.upper()
    df.loc[i[150:300], "kota"] = " " + df.loc[i[150:300], "kota"].str.lower() + " "
    bdg = df.index[df["kota"] == "Bandung"][:60]
    df.loc[bdg, "kota"] = "Bdg"
    df.loc[rng.choice(n, 25, replace=False), "umur"] = np.nan
    df.loc[rng.choice(n, 8, replace=False), "umur"] = rng.choice([0, 150, 999], 8)
    df.loc[rng.choice(n, 40, replace=False), "biaya_bulanan"] = np.nan
    df.loc[rng.choice(n, 30, replace=False), "metode_bayar"] = np.nan
    df = pd.concat([df, df.sample(30, random_state=seed)], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df_churn = buat_data_churn()

**a.** Tampilkan jumlah baris dan kolom `df_churn`, informasi tipe data (`info()`), dan 5 baris pertama. Lalu tampilkan
jumlah serta persentase pelanggan `churn` = Ya dan Tidak, beserta perbandingannya dalam **pie chart**.
✍️ Apakah kelasnya seimbang? Metrik apa yang sebaiknya diperhatikan selain accuracy?

*Show the shape, info(), and the first 5 rows. Then show the count and percentage of each churn class with a pie chart.
✍️ Are the classes balanced? Which metric should be watched besides accuracy?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**b.** Lakukan *data cleaning* dan simpan hasilnya di `df1`:
1. Tampilkan jumlah baris duplikat, lalu hapus.
2. Hapus kolom `customer_id`.
3. Seragamkan `kota` (hapus spasi, huruf kapital di awal kata, `Bdg` → `Bandung`), lalu tampilkan `value_counts()`.
4. Ubah `umur` yang mustahil (< 17 atau > 90) menjadi `NaN`.
5. Ubah `churn` menjadi biner (Ya = 1, Tidak = 0).
6. Tampilkan jumlah missing value per kolom. **Jangan isi** missing value di sini (akan diisi di dalam `Pipeline`).

✍️ Mengapa pengisian missing value sebaiknya dilakukan di dalam `Pipeline` setelah split, bukan sebelum split?

*Clean the data into df1: drop duplicates, drop customer_id, standardize kota, set impossible ages to NaN, encode
churn as 1/0, and show missing values per column (do not impute yet).
✍️ Why should imputation happen inside the Pipeline after the split?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**c.** Eksplorasi faktor churn:
1. Buat 1 figure 1×3: **tingkat churn (%) per `kontrak`** (bar), **tingkat churn (%) per `metode_bayar`** (bar), dan
   **boxplot `lama_berlangganan` per kelas churn**.
2. Tampilkan korelasi setiap kolom numerik dengan `churn`, diurutkan.

✍️ Sebutkan 3 temuan terpenting.

*Plot the churn rate per contract type and per payment method, and a boxplot of tenure per churn class (1×3 figure).
Show the correlation of each numeric column with churn. ✍️ Write the 3 most important findings.*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**d.** Siapkan data untuk model:
1. Pisahkan `X` (semua kolom kecuali `churn`) dan `y`. Bagi **80:20** dengan `stratify=y` dan `random_state=42`.
   Tampilkan ukuran dan proporsi churn di train dan test.
2. Buat daftar `num_cols` dan `cat_cols`.
3. Buat fungsi `buat_preprocessor(scale=True)` yang mengembalikan `ColumnTransformer` **baru**: numerik →
   `SimpleImputer(median)` (+ `StandardScaler` jika `scale=True`); kategorikal → `SimpleImputer(most_frequent)` +
   `OneHotEncoder(handle_unknown="ignore")`.

*Split 80:20 with stratify, list numeric and categorical columns, and write buat_preprocessor(scale=True) that returns
a new ColumnTransformer (median imputer + optional scaler for numeric; mode imputer + one-hot for categorical).*

In [ ]:
# TODO

**e.** Latih tiga model dalam `Pipeline` (`buat_preprocessor()` + model) dan simpan di dict `models_1`:
- `LogisticRegression(max_iter=1000)`
- `KNeighborsClassifier(n_neighbors=15)`
- `RandomForestClassifier(n_estimators=300, random_state=42)` (preprocessor tanpa scaling)

Hitung **accuracy, precision, recall, F1, dan ROC-AUC** pada data test, simpan dalam DataFrame `hasil_1`
(index = nama model). ✍️ Model mana yang paling cocok untuk tim retensi dan mengapa?

*Train Logistic Regression, KNN (k=15), and Random Forest pipelines, store them in models_1, and build hasil_1 with
accuracy, precision, recall, F1, and ROC-AUC on the test set. ✍️ Which model suits the retention team best, and why?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**f.** 1. Tampilkan **confusion matrix** Regresi Logistik dalam bentuk heatmap (label `Tidak`/`Ya`).
2. Turunkan **threshold** Regresi Logistik dari 0.5 menjadi 0.3 (`predict_proba(X_test)[:, 1] >= 0.3`).
   Bandingkan precision, recall, F1, dan jumlah pelanggan yang ditandai churn untuk kedua threshold dalam DataFrame
   `hasil_threshold`.

✍️ Jelaskan *trade-off* yang terjadi dan threshold mana yang Anda sarankan.

*Plot the Logistic Regression confusion matrix as a heatmap, then compare thresholds 0.5 and 0.3 (precision, recall,
F1, number of flagged customers) in hasil_threshold. ✍️ Explain the trade-off and recommend a threshold.*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**g.** Interpretasi model:
1. Ambil nama fitur hasil preprocessing (`get_feature_names_out()`), lalu tampilkan **5 koefisien Regresi Logistik**
   dengan nilai absolut terbesar.
2. Tampilkan **5 feature importance** teratas Random Forest.
3. Buat 1 figure 1×2 berisi kedua bar chart tersebut.

✍️ Jelaskan arti tanda koefisien dan apakah kedua model sepakat tentang fitur terpenting.

*Show the 5 Logistic Regression coefficients with the largest absolute values and the top 5 Random Forest feature
importances, as two bar charts in one 1×2 figure. ✍️ Explain the sign of the coefficients and whether both models agree.*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**h.** **(BONUS)** Latih ulang Regresi Logistik dengan `class_weight="balanced"`. Bandingkan precision, recall, dan F1-nya
dengan Regresi Logistik biasa. ✍️ Apa efek `class_weight="balanced"`?

*(BONUS) Retrain Logistic Regression with class_weight="balanced" and compare precision, recall, and F1 with the
default model. ✍️ What does class_weight="balanced" do?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

---
## SOAL 2 — Klasifikasi Wine dengan KNN dan ANN (25 poin)

Dataset bawaan scikit-learn **wine** berisi 13 hasil analisis kimia dari 178 wine yang berasal dari 3 kultivar
(`class_0`, `class_1`, `class_2`). Skala fiturnya sangat berbeda, misalnya `proline` ratusan sampai ribuan,
sedangkan `hue` di bawah 2.

*The built-in wine dataset has 13 chemical measurements of 178 wines from 3 cultivars. Feature scales differ a lot.*

### Setup dataset (jangan diubah)

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)

**a.** Buat DataFrame `df_wine` dari `wine.frame`, tambahkan kolom `kelas` berisi nama kelas (`wine.target_names`).
Tampilkan ukuran data, jumlah data per kelas, serta nilai **min, max, mean, dan std** setiap fitur.
✍️ Fitur mana yang skalanya paling besar dan paling kecil? Apa akibatnya untuk KNN?

*Create df_wine with a kelas column of class names. Show the shape, class counts, and min/max/mean/std of each feature.
✍️ Which features have the largest and smallest scale, and what does that mean for KNN?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**b.** Pisahkan `X_w` (13 fitur) dan `y_w` (`target`), lalu bagi **75:25** dengan `stratify` dan `random_state=42`.
Tampilkan ukuran train dan test.

*Split X_w and y_w 75:25 with stratify and random_state=42, and show the shapes.*

In [ ]:
# TODO

**c.** Latih `KNeighborsClassifier(n_neighbors=5)` **tanpa scaling** dan **dengan** `StandardScaler` (dalam `Pipeline`).
Tampilkan accuracy test keduanya. ✍️ Jelaskan perbedaannya.

*Train KNN (k=5) without scaling and with StandardScaler in a Pipeline, and show both test accuracies. ✍️ Explain.*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**d.** Cari *k* terbaik untuk KNN dengan scaling: untuk *k* = 1, 3, 5, …, 29 hitung rata-rata accuracy
**5-fold `StratifiedKFold`** (`shuffle=True`, `random_state=42`) **pada data train**. Simpan di Series `cv_k`,
buat line plot *k* vs accuracy CV, lalu latih ulang model dengan *k* terbaik (`best_k`) dan tampilkan accuracy test.
Jika beberapa *k* memiliki skor CV yang sama, pilih *k* **terbesar** (model lebih stabil).
✍️ Mengapa *k* dipilih dengan CV di data train, bukan dengan data test?

*For k = 1, 3, …, 29, compute the mean 5-fold stratified CV accuracy on the training data (with scaling), plot it,
retrain with the best k (largest k on ties), and show the test accuracy. ✍️ Why choose k with CV on train instead of the test set?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**e.** Buat **ANN** dengan `Pipeline` `StandardScaler` → `MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000,
random_state=42)` dan simpan di `ann_wine`. Tampilkan accuracy train dan test, jumlah epoch (`n_iter_`), lalu plot
**kurva loss** (`loss_curve_`).

*Build an ANN pipeline (StandardScaler → MLPClassifier((32, 16), max_iter=2000, random_state=42)) as ann_wine.
Show train/test accuracy and the number of epochs, and plot the loss curve.*

In [ ]:
# TODO

**f.** Lakukan `GridSearchCV` (cv = `skf`, scoring accuracy) pada pipeline ANN dengan grid:
`hidden_layer_sizes` ∈ {(16,), (32, 16), (64, 32)}, `alpha` ∈ {0.0001, 0.01, 1}, `learning_rate_init` ∈ {0.001, 0.01}.
Tampilkan parameter terbaik, skor CV terbaik, dan accuracy test model terbaik (`ann_best`).
✍️ Apa fungsi parameter `alpha`?

*Run GridSearchCV on the ANN pipeline with the given grid. Show the best parameters, best CV score, and the test
accuracy of the best model (ann_best). ✍️ What does alpha do?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**g.** Buat DataFrame `hasil_2` berisi accuracy test untuk: KNN tanpa scaling, KNN dengan scaling (k=5), KNN *k* terbaik,
ANN (32, 16), dan ANN hasil GridSearch. Tampilkan juga `classification_report` model terbaik.
✍️ Model mana yang Anda pilih? Mengapa scaling penting untuk KNN **dan** ANN?

*Build hasil_2 with the test accuracy of the five models and show the classification report of the best one.
✍️ Which model do you choose? Why is scaling important for both KNN and ANN?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**h.** **(BONUS)** Dengan dua fitur saja, `alcohol` dan `proline`, gambarkan **decision boundary** KNN (*k* = 15) tanpa scaling
dan dengan scaling dalam 1 figure 1×2 (`DecisionBoundaryDisplay`). ✍️ Apa yang terlihat?

*(BONUS) Using only alcohol and proline, draw the KNN (k=15) decision boundary without and with scaling (1×2 figure).
✍️ What do you see?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

---
## SOAL 3 — Diagnosis Kanker Payudara dengan Ensemble (25 poin)

Dataset bawaan **breast cancer** berisi 30 fitur hasil citra sel dari 569 pasien, dengan diagnosis *malignant*
(ganas) atau *benign* (jinak). Pada dataset ini `target = 0` berarti **malignant**. Kesalahan paling berbahaya adalah
pasien ganas yang diprediksi jinak (**false negative**).

*The built-in breast cancer dataset has 30 features from 569 patients. target = 0 means malignant. The most dangerous
error is a malignant tumour predicted as benign (false negative).*

### Setup dataset (jangan diubah)

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

**a.** Buat `df_bc` dari `bc.frame`, lalu buat kolom `ganas` = 1 jika `target == 0` (malignant) dan 0 jika sebaliknya.
Tampilkan jumlah dan persentase setiap kelas `ganas`. ✍️ Mengapa kelas positif (1) sebaiknya adalah *malignant*?

*Create df_bc and a column ganas = 1 for malignant (target == 0). Show the count and percentage of each class.
✍️ Why should malignant be the positive class (1)?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**b.** Pisahkan `X_b` (30 fitur) dan `y_b` (`ganas`), bagi **80:20** dengan `stratify` dan `random_state=42`.
Buat fungsi `metrik(model)` yang mengembalikan dict **accuracy, precision, recall, dan F1** model pada data test.

*Split X_b and y_b 80:20 with stratify and random_state=42. Write metrik(model) that returns a dict of test accuracy,
precision, recall, and F1.*

In [ ]:
# TODO

**c.** Buat tiga **model dasar** dan simpan di list `base`:
- `("lr", Pipeline(StandardScaler, LogisticRegression(max_iter=1000)))`
- `("knn", Pipeline(StandardScaler, KNeighborsClassifier(n_neighbors=7)))`
- `("tree", DecisionTreeClassifier(max_depth=4, random_state=42))`

Untuk tiap model, hitung rata-rata **accuracy dan recall 5-fold CV** di data train (`cross_validate`, cv = `skf`),
simpan di DataFrame `cv_base`.

*Create the three base models in the list base and compute their mean 5-fold CV accuracy and recall on the training
data with cross_validate, stored in cv_base.*

In [ ]:
# TODO

**d.** Gabungkan ketiga model dasar dengan `VotingClassifier`, versi **hard** (`vote_hard`) dan **soft** (`vote_soft`).
Latih di data train dan tampilkan `metrik()` keduanya beserta ketiga model dasar dalam satu DataFrame `hasil_vote`.
✍️ Apa beda hard dan soft voting?

*Combine the base models with VotingClassifier (hard and soft), train them, and show metrik() for the two voting models
and the three base models in hasil_vote. ✍️ What is the difference between hard and soft voting?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**e.** Bandingkan **Decision Tree tunggal** (`max_depth=None`, `random_state=42`) dengan ensemble berbasis pohon:
- `BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=42)`
- `RandomForestClassifier(n_estimators=300, random_state=42)`
- `AdaBoostClassifier(n_estimators=200, random_state=42)`
- `GradientBoostingClassifier(random_state=42)`

Tampilkan accuracy **train** dan `metrik()` test dalam DataFrame `hasil_tree`. ✍️ Jelaskan perbedaan bagging dan
boosting berdasarkan hasilnya.

*Compare a single fully grown Decision Tree with Bagging, Random Forest, AdaBoost, and Gradient Boosting. Show the train
accuracy and the test metrics in hasil_tree. ✍️ Explain bagging vs boosting using the results.*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**f.** Buat `StackingClassifier` dengan `estimators = base + [("rf", RandomForestClassifier(n_estimators=300,
random_state=42))]`, `final_estimator=LogisticRegression(max_iter=1000)`, dan `cv=5`. Tampilkan `metrik()`-nya.
✍️ Mengapa stacking melatih *meta-model* dari prediksi *out-of-fold*?

*Build a StackingClassifier (base models + Random Forest, final LogisticRegression, cv=5) and show its metrics.
✍️ Why does stacking train the meta-model on out-of-fold predictions?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**g.** Gabungkan semua hasil Soal 3 (model dasar, voting, model pohon, stacking) ke DataFrame `hasil_3`, urutkan berdasarkan
**recall** lalu F1, dan buat **bar chart** recall dan F1 per model. ✍️ Model mana yang Anda rekomendasikan untuk skrining
kanker dan mengapa?

*Combine all Soal 3 results in hasil_3, sort by recall then F1, and plot recall and F1 per model.
✍️ Which model do you recommend for cancer screening, and why?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

**h.** **(BONUS)** Tampilkan 10 fitur terpenting Random Forest berdasarkan `feature_importances_` dan berdasarkan
**permutation importance** pada data test (`n_repeats=10`, `random_state=42`) dalam 1 figure 1×2.
✍️ Mengapa hasil keduanya bisa berbeda?

*(BONUS) Show the top 10 Random Forest features by feature_importances_ and by permutation importance on the test set
(1×2 figure). ✍️ Why can the two rankings differ?*

In [ ]:
# TODO

✍️ **Jawaban:**

_tulis jawaban di sini_

---
## SOAL 4 — Teori (10 poin)

Jawab singkat di sel teks di bawah setiap pertanyaan. *Answer briefly in the text cell below each question.*

**1.** Jelaskan perbedaan **bagging** dan **boosting**, beserta contoh model masing-masing.

**Jawaban:** _tulis jawaban di sini_

**2.** Mengapa `StandardScaler` dan `SimpleImputer` harus di-*fit* hanya pada data train?

**Jawaban:** _tulis jawaban di sini_

**3.** Apa fungsi `stratify=y` pada `train_test_split`?

**Jawaban:** _tulis jawaban di sini_

**4.** Apa pengaruh nilai *k* yang terlalu kecil dan terlalu besar pada KNN?

**Jawaban:** _tulis jawaban di sini_

**5.** Mengapa Random Forest tidak memerlukan scaling, sedangkan KNN, Regresi Logistik, dan ANN memerlukannya?

**Jawaban:** _tulis jawaban di sini_

**6.** Kapan accuracy menyesatkan? Beri contoh kapan *recall* lebih penting dan kapan *precision* lebih penting.

**Jawaban:** _tulis jawaban di sini_

**7.** Jelaskan peran fungsi aktivasi, `alpha`, dan `early_stopping` pada `MLPClassifier`.

**Jawaban:** _tulis jawaban di sini_

**8.** Apa beda *hard voting* dan *soft voting*, dan apa syarat soft voting?

**Jawaban:** _tulis jawaban di sini_

**9.** Apa itu *OOB score* pada Random Forest?

**Jawaban:** _tulis jawaban di sini_

**10.** Mengapa hasil koefisien Regresi Logistik baru bisa dibandingkan antar fitur setelah scaling?

**Jawaban:** _tulis jawaban di sini_